# TP – Systèmes de recommandation

**Cours** : INF5063 – Machine learning : applications (G. Nollet, L. Benedetti)
**Auteur** : Alban Rouault

Objectif : concevoir et évaluer un système de recommandation de films par filtrage collaboratif
*user-based* sur le jeu de données Kaggle
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset).

## Imports et configuration

Toutes les dépendances sont déclarées dans `pyproject.toml` et gérées avec `uv`
(`uv run jupyter lab` pour lancer le notebook).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Affichage des DataFrames
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Reproductibilité (séparation train/test, tirages aléatoires)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Emplacement des données
DATA_DIR = Path("Ressources/dataset")

## Exercice 1 – Chargement des données

### 1) Chargement des fichiers

> Récupérer sur Kaggle et charger les fichiers suivants du dataset : `movies_metadata.csv`, `ratings_small.csv` et `links_small.csv`.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "ratings_small.csv")
links = pd.read_csv(DATA_DIR / "links_small.csv")

tables = {"movies_metadata": movies, "ratings_small": ratings, "links_small": links}
for name, df in tables.items():
    print(f"{name:<16} {df.shape[0]:>7,} lignes  x {df.shape[1]:>2} colonnes")

**Réponse.** Les trois fichiers viennent de
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) (Kaggle) et sont
dans `Ressources/dataset/`. `movies_metadata.csv` est lu avec `low_memory=False` : quelques lignes
mal formées mélangent les types, on les traite à la question 5.

### 2) Aperçu des tables

> Afficher un aperçu de chaque table et vérifier leur compréhension.

In [ ]:
for name, df in tables.items():
    print(f"── {name} : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    display(df.head(3))

In [ ]:
def resume_colonnes(df: pd.DataFrame) -> pd.DataFrame:
    """Type, remplissage et cardinalité de chaque colonne."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "% nuls": (df.isna().mean() * 100).round(1),
        "valeurs distinctes": df.nunique(),
        "exemple": df.iloc[0].astype(str).str.slice(0, 40),
    })

resume_colonnes(movies)

In [ ]:
print("Valeurs de rating :", sorted(ratings["rating"].unique().tolist()))
print("Période des avis  :",
      pd.to_datetime(ratings["timestamp"].min(), unit="s").date(), "->",
      pd.to_datetime(ratings["timestamp"].max(), unit="s").date())
print("Nuls dans ratings :", ratings.isna().sum().sum(), "| nuls dans links :", links.isna().sum().to_dict())

**Réponse.**

**`movies_metadata`** – une ligne par film du catalogue TMDB (45 466 lignes).

| Colonne | Contenu |
|---|---|
| `id` | identifiant TMDB du film (clé) |
| `imdb_id` | identifiant IMDB (`tt0114709`) |
| `title`, `original_title` | titre (anglais) et titre original |
| `overview`, `tagline` | synopsis et phrase d'accroche |
| `genres` | liste de genres, stockée comme texte JSON |
| `release_date` | date de sortie |
| `runtime` | durée en minutes |
| `budget`, `revenue` | budget et recettes mondiales en dollars (0 = inconnu) |
| `popularity` | score de popularité TMDB |
| `vote_average`, `vote_count` | note moyenne sur 10 et nombre de votes sur TMDB |
| `original_language`, `spoken_languages` | langue d'origine (code ISO) et langues parlées |
| `production_companies`, `production_countries` | sociétés et pays de production (texte JSON) |
| `belongs_to_collection` | saga du film (Toy Story Collection…), vide dans 90 % des cas |
| `status` | Released, Post Production, Rumored… |
| `adult`, `video` | film pour adultes ; vidéo hors salle (rares) |
| `homepage`, `poster_path` | site officiel et chemin de l'affiche |

**`ratings_small`** – une ligne par avis (100 004 lignes).

| Colonne | Contenu |
|---|---|
| `userId` | identifiant de l'utilisateur (MovieLens) |
| `movieId` | identifiant du film (MovieLens, **différent** de l'id TMDB) |
| `rating` | note de 0,5 à 5 par pas de 0,5 |
| `timestamp` | date de l'avis (secondes Unix, de 1995 à 2016) |

**`links_small`** – une ligne par film du jeu réduit (9 125 lignes).

| Colonne | Contenu |
|---|---|
| `movieId` | identifiant MovieLens (celui de `ratings_small`) |
| `imdbId` | identifiant IMDB |
| `tmdbId` | identifiant TMDB (celui de `movies_metadata`), manquant pour 13 films |

À retenir : `ratings_small` est propre (aucun nul), alors que `movies_metadata` a des colonnes très
peu remplies et des colonnes numériques lues comme du texte (`id`, `budget`, `popularity`) à cause de
lignes mal formées.

### 3) Effectifs

> Combien d'utilisateurs y a-t-il ? De films ? D'avis ?

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
n_ratings = len(ratings)
avis_par_user = ratings.groupby("userId").size()

pd.DataFrame({
    "valeur": [n_users, n_ratings, n_movies_rated, links["movieId"].nunique(), movies["id"].nunique(),
               f"{avis_par_user.min()} / {avis_par_user.median():.0f} / {avis_par_user.max()}",
               f"{n_ratings / (n_users * n_movies_rated):.2%}"],
}, index=["utilisateurs", "avis", "films notés", "films dans links_small", "films dans movies_metadata",
          "avis par utilisateur (min / médiane / max)", "densité de la matrice utilisateurs x films"])

**Réponse.**

| | |
|---|---|
| Utilisateurs | **671** |
| Avis | **100 004** |
| Films notés | **9 066** (sur 9 125 dans `links_small` et 45 436 dans le catalogue) |

Chaque utilisateur a noté au moins 20 films (médiane 71). La matrice utilisateurs × films n'est remplie
qu'à **1,6 %** : elle est très creuse, ce sera la difficulté principale du filtrage collaboratif.

### 4) Rôle de `links_small.csv`

> Quelle est l'utilité du fichier `links_small.csv` par rapport aux deux autres fichiers ?

In [ ]:
# Le film movieId = 1 dans ratings : quel est-il ?
tmdb_id = int(links.loc[links["movieId"] == 1, "tmdbId"].iloc[0])
print(f"ratings movieId = 1  ->  links tmdbId = {tmdb_id}  ->  movies_metadata :")
display(movies.loc[movies["id"] == str(tmdb_id), ["id", "imdb_id", "title", "release_date"]])

print("movieId de ratings absents de links :", (~ratings["movieId"].isin(links["movieId"])).sum())
print("Films de links sans tmdbId          :", links["tmdbId"].isna().sum())
print("tmdbId de links absents de metadata :",
      (~links["tmdbId"].dropna().astype(int).astype(str).isin(movies["id"])).sum())

**Réponse.** Les avis utilisent des identifiants **MovieLens** (`movieId`), les métadonnées des
identifiants **TMDB** (`id`) : les deux tables n'ont aucune clé commune. `links_small` est la **table
de correspondance** qui permet de passer de l'un à l'autre, donc de retrouver le titre d'un film noté
(`movieId` 1 → `tmdbId` 862 → *Toy Story*).

Limite : 13 films n'ont pas de `tmdbId` et 30 `tmdbId` sont absents du catalogue. Ces films pourront
être notés et recommandés, mais pas affichés avec leur titre.

### 5) Nettoyage de `movies_metadata`

> Nettoyer la table `movies_metadata.csv` : convertir tous les ID de films au format numérique (en retirant les films dont les ID ne sont pas numériques), ne conserver qu'une ligne par ID unique.

Le nettoyage se fait en mémoire : le CSV n'est jamais modifié. On garde `movies` (brut) et on crée `movies_clean`.

In [ ]:
# 1. ID au format numérique : les valeurs non convertibles deviennent NaN
id_num = pd.to_numeric(movies["id"], errors="coerce")
print(f"Lignes dont l'id n'est pas numérique : {id_num.isna().sum()}")
display(movies.loc[id_num.isna(), ["adult", "budget", "id", "title", "release_date", "popularity", "revenue"]])

Ces trois lignes sont **décalées** : un morceau de synopsis dans `adult`, un chemin d'affiche dans
`budget`, une date dans `id`, pas de titre. Elles sont inexploitables.

In [ ]:
# 2. Retrait de ces lignes et conversion en entier
movies_clean = movies.loc[id_num.notna()].copy()
movies_clean["id"] = id_num.loc[id_num.notna()].astype(int)

# 3. Doublons sur l'id
doublons = movies_clean[movies_clean.duplicated("id", keep=False)].sort_values("id")
print(f"Lignes en doublon sur l'id : {doublons['id'].duplicated().sum()} "
      f"({doublons['id'].nunique()} ids concernés, {doublons.duplicated().sum()} doublons strictement identiques)")

# Deux exemples : un doublon identique (105045) et un doublon qui diffère (4912)
display(doublons.loc[doublons["id"].isin([105045, 4912]),
                     ["id", "title", "release_date", "popularity", "vote_count", "revenue"]])

cols_diff = {c for _, g in doublons.groupby("id") for c in g.columns if g[c].astype(str).nunique() > 1}
print("Colonnes qui diffèrent entre doublons :", cols_diff)

In [ ]:
movies_clean = movies_clean.drop_duplicates("id", keep="first").reset_index(drop=True)

print(f"Avant : {len(movies):,} lignes  ->  après : {len(movies_clean):,} lignes")
print("id unique :", movies_clean["id"].is_unique, "| dtype :", movies_clean["id"].dtype)

**Réponse.** Deux opérations :

1. **3 lignes** ont un `id` non numérique (lignes décalées de l'export) : retirées, puis `id` converti en entier.
2. **30 lignes** sont en doublon (29 films). Elles ne diffèrent que par `popularity`, un score qui bouge
   dans le temps : ce sont deux exports du même film. On garde la première.

Résultat : **45 433 films**, un par `id` entier, joignable directement à `tmdbId` de `links_small`.

### 6) Nettoyage de `ratings_small`

> Nettoyer la table `ratings_small.csv` : retirer les avis manquants et les ID d'utilisateurs et de films non conformes, et convertir tous les ID d'utilisateurs et de films au format numérique.

Un avis est conforme si : la note est présente et sur l'échelle MovieLens (0,5 à 5 par pas de 0,5),
les identifiants sont des entiers strictement positifs, et le couple (utilisateur, film) est unique.

In [ ]:
n_avant = len(ratings)
ratings_clean = ratings.copy()

# 1. Avis manquants
manquants = ratings_clean[["userId", "movieId", "rating"]].isna().any(axis=1)
print(f"Avis avec valeur manquante          : {manquants.sum()}")
ratings_clean = ratings_clean[~manquants]

# 2. Identifiants conformes -> entiers
for col in ["userId", "movieId"]:
    num = pd.to_numeric(ratings_clean[col], errors="coerce")
    conforme = num.notna() & (num > 0) & (num % 1 == 0)
    print(f"{col:<8} non conformes               : {(~conforme).sum()}")
    ratings_clean = ratings_clean[conforme].assign(**{col: num[conforme].astype(int)})

# 3. Notes conformes
note_ok = ratings_clean["rating"].between(0.5, 5) & ((ratings_clean["rating"] * 2) % 1 == 0)
print(f"Notes hors échelle MovieLens        : {(~note_ok).sum()}")
ratings_clean = ratings_clean[note_ok]

# 4. Doublons (utilisateur, film)
dup = ratings_clean.duplicated(["userId", "movieId"])
print(f"Doublons (userId, movieId)          : {dup.sum()}")
ratings_clean = ratings_clean[~dup].reset_index(drop=True)

# Films sans métadonnées (conservés : le filtrage collaboratif n'en a pas besoin)
sans_meta = ~ratings_clean["movieId"].isin(links.loc[links["tmdbId"].isin(movies_clean["id"]), "movieId"])
print(f"Avis sur un film sans métadonnées   : {sans_meta.sum()} (conservés)")

print(f"\nAvant : {n_avant:,} avis  ->  après : {len(ratings_clean):,} avis")
print(ratings_clean.dtypes.to_string())

**Réponse.** `ratings_small` est déjà propre : aucun avis manquant, aucun identifiant non conforme,
aucun doublon, et les identifiants sont déjà des entiers. Le code reste défensif pour fonctionner sur
un fichier moins propre. Les **100 004 avis** sont conservés, y compris les 194 qui portent sur un film
sans métadonnées : le filtrage collaboratif n'utilise que les notes.

### 7) Les 10 films les plus anciens

> Déterminer les 10 films les plus anciens du dataset.

In [ ]:
# Conversion de release_date en date ; les valeurs invalides deviennent NaT
movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")

print("Films sans date de sortie :", movies_clean["release_date"].isna().sum())
futur = movies_clean[movies_clean["release_date"] > "2017-12-31"]
print(f"Films datés après 2017 (extraction du dataset) : {len(futur)}")
display(futur[["id", "title", "release_date", "status", "vote_count"]])

In [ ]:
cols = ["id", "title", "release_date", "runtime", "original_language", "vote_count"]
movies_clean.dropna(subset=["release_date"]).nsmallest(10, "release_date")[cols]

In [ ]:
# Même question restreinte aux films effectivement notés dans ratings_small
tmdb_notes = links.loc[links["movieId"].isin(ratings_clean["movieId"]), "tmdbId"].dropna().astype(int)
films_notes = movies_clean[movies_clean["id"].isin(tmdb_notes)]
print(f"Films notés retrouvés dans les métadonnées : {len(films_notes):,}")
films_notes.nsmallest(10, "release_date")[cols]

**Réponse.** Anomalies repérées avant de trier : 87 films sans date, et 6 films datés de 2018 ou
2020, donc pas encore sortis quand le dataset a été extrait (2017).

Les 10 films les plus anciens du catalogue vont de **1874** (*Passage of Venus*) à **1890** : des films
d'une minute des pionniers du cinéma. Parmi les films réellement notés dans `ratings_small`, le plus
ancien est *A Trip to the Moon* (1902), suivi de muets des années 1915 à 1921.

### 8) Les 10 plus gros succès au box-office

> Déterminer les 10 films ayant eu le plus gros succès au box-office (colonne `revenue`).

In [ ]:
movies_clean["revenue"] = pd.to_numeric(movies_clean["revenue"], errors="coerce")
rev = movies_clean["revenue"]

print(f"revenue manquant (NaN)  : {rev.isna().sum():>6,}")
print(f"revenue = 0             : {(rev == 0).sum():>6,}  ({(rev == 0).mean():.1%} des films)")
print(f"0 < revenue < 1 000     : {rev.between(1, 999).sum():>6,}")
print(f"revenue >= 1 000        : {(rev >= 1000).sum():>6,}")

# Exemples de valeurs suspectes : quelques dollars de recettes pour des films sortis en salle
display(movies_clean.loc[rev.between(1, 999), ["title", "release_date", "budget", "revenue"]].head(5))

In [ ]:
top_revenue = movies_clean.nlargest(10, "revenue").copy()
top_revenue["revenue (M$)"] = (top_revenue["revenue"] / 1e6).round(0)
top_revenue["budget (M$)"] = (pd.to_numeric(top_revenue["budget"], errors="coerce") / 1e6).round(0)
top_revenue[["id", "title", "release_date", "revenue (M$)", "budget (M$)", "vote_average", "vote_count"]]

**Réponse.** La colonne `revenue` est très incomplète : **84 % des films ont 0**, qui veut dire
"inconnu", et 154 films affichent quelques dollars (unité fausse). Cela ne gêne pas le classement des
plus gros succès, qui ne regarde que les grandes valeurs.

Le top 10 va d'**Avatar** (2,79 milliards de dollars) à **Beauty and the Beast** (1,26), en passant par
*Star Wars : The Force Awakens*, *Titanic*, *The Avengers*, *Jurassic World* et *Furious 7*. Les recettes
sont en dollars courants, non corrigées de l'inflation, d'où la domination des films récents.

### 9) Séparation entraînement / test

> Nous aurons besoin d'une séparation entraînement/test pour tester nos modèles. Que faut-il séparer en deux ? Effectuer une séparation entraînement/test de sorte à ce que chaque utilisateur en test ait une chance de recevoir de bonnes recommandations. Comment avez-vous fait ?

In [ ]:
SEUIL_PERTINENT = 4.0   # un film noté >= 4 est considéré comme "aimé" (servira aussi pour les métriques top k)
FRAC_TEST = 0.2


def split_par_utilisateur(ratings: pd.DataFrame, frac_test: float = FRAC_TEST,
                          seuil_pertinent: float = SEUIL_PERTINENT,
                          seed: int = RANDOM_STATE) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Sépare les AVIS en train/test, utilisateur par utilisateur.

    Pour chaque utilisateur, frac_test de ses avis partent en test, en stratifiant sur
    "film aimé / non aimé" (au moins un avis par strate). Chaque utilisateur garde ainsi
    la majorité de ses avis en train et a au moins un film aimé en test.
    """
    # Un seul générateur pour tous les groupes : réutiliser la même graine dans chaque
    # groupe ferait tirer les mêmes positions pour tous les groupes de même taille.
    rng = np.random.default_rng(seed)
    aime = ratings["rating"] >= seuil_pertinent
    strate = ratings["userId"].astype(str) + "_" + aime.astype(str)

    def tirer(groupe):
        n_test = max(1, round(len(groupe) * frac_test))   # au moins 1 avis par strate
        return groupe.sample(n=n_test, random_state=rng)

    test = ratings.groupby(strate, group_keys=False).apply(tirer, include_groups=False)
    train = ratings.drop(test.index)
    return train.reset_index(drop=True), test.reset_index(drop=True)


train, test = split_par_utilisateur(ratings_clean)
print(f"train : {len(train):>6,} avis ({len(train) / len(ratings_clean):.1%})")
print(f"test  : {len(test):>6,} avis ({len(test) / len(ratings_clean):.1%})")

In [ ]:
# Vérifications : chaque utilisateur est-il bien "testable" ?
par_user_train = train.groupby("userId").size()
par_user_test = test.groupby("userId").size()
aimes_test = test[test["rating"] >= SEUIL_PERTINENT].groupby("userId").size()

print(f"Utilisateurs en train / en test          : {train['userId'].nunique()} / {test['userId'].nunique()}")
print(f"Avis par utilisateur en train (min/méd.) : {par_user_train.min()} / {par_user_train.median():.0f}")
print(f"Avis par utilisateur en test  (min/méd.) : {par_user_test.min()} / {par_user_test.median():.0f}")
print(f"Utilisateurs sans film aimé en test      : {test['userId'].nunique() - len(aimes_test)}")
print(f"Films aimés en test par utilisateur (méd.): {aimes_test.median():.0f}")
print("Avis communs train/test                  :", len(train.merge(test, on=["userId", "movieId"])))

absents = ~test["movieId"].isin(train["movieId"])
print(f"Avis de test sur un film absent du train : {absents.sum()} ({absents.mean():.1%})")

**Réponse.**

**On sépare les avis**, pas les utilisateurs. Pour recommander à quelqu'un, le modèle doit connaître
une partie de ses goûts : on lui cache donc 20 % des avis de chaque utilisateur, et on regarde s'il
les retrouve.

**Comment.** Le tirage est fait **utilisateur par utilisateur** (un tirage global pourrait laisser un
utilisateur avec presque rien en train), et **stratifié sur "film aimé (note ≥ 4) / non aimé"** avec au
moins un avis par strate : chaque utilisateur a ainsi au moins un film aimé en test, sans quoi aucune
recommandation ne pourrait être jugée bonne. Un seul générateur aléatoire, initialisé avec
`RANDOM_STATE`, sert à tous les utilisateurs : le tirage est reproductible et vraiment aléatoire.

**Résultat.** 79 976 avis en train, 20 028 en test. Les 671 utilisateurs sont des deux côtés, avec au
moins 15 avis en train et 4 en test, et tous ont au moins un film aimé en test. Aucun avis en commun.

**Limite.** 745 avis de test (3,7 %) concernent des films que personne n'a notés en train : le filtrage
collaboratif ne pourra pas les prédire, on le gérera à l'évaluation. Une séparation temporelle (derniers
avis en test) serait plus réaliste mais n'est pas demandée ici.